In [9]:
import requests
import pandas as pd
import json
import re

def search_epmc(query, page_size=100, sort_by=""): #relevance #CITED desc
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    params = {
        "query": query,
        "format": "json",
        "pageSize": page_size,
        'synonym': 'FALSE',
        'resultType': 'core',
        "sort": sort_by
    }
    response = requests.get(url, params=params)
    return response.json()['resultList']['result']

def get_references(pmid):
    url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/MED/{pmid}/references"
    params = {"format": "json",
              "pageSize":200}
    response = requests.get(url, params=params)
    return response.json().get('referenceList', {}).get('reference', [])

def extract_database_references(references):
    db_related = []
    keywords = ["database", "resource", "platform", "repository", "portal"]
    for ref in references:
        if any(kw in ref.get('title', '').lower() for kw in keywords):
            db_related.append({
                "title": ref.get("title", ""),
                "journal": ref.get("journalTitle", ""),
                "year": ref.get("pubYear", ""),
                "author": ref.get("authorString", ""),
            })
    return db_related

In [10]:

# 提取信息
data = []

recent_10_year_list = [2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015]

for year in recent_10_year_list:
    articles = search_epmc(f'"splicing" AND "ribosome" AND PUB_YEAR:[{year-1} TO {year}] ', page_size= 200)
    for article in articles:
        data.append({
            'title': article.get('title', ''),
            'keywordList':article.get('keywordList', ''),
            'web':'https://doi.org/'+article.get('doi', ''),
            'pubYear': article.get('pubYear', ''),
            'journal': article.get('journalInfo', '').get('journal', '').get('title', ''),
            'citedByCount': article.get('citedByCount', ''),
            'pmid': article.get('pmid', ''),
            'pmcid': article.get('pmcid', ''),
            'source': article.get('source', ''),
            'doi': article.get('doi', ''),
            'abstractText': article.get('abstractText', ''),
            
        })

df_relevance = pd.DataFrame(data)
df_relevance['citedByCount'] = pd.to_numeric(df_relevance['citedByCount'], errors='coerce').fillna(0).astype(int)
df_relevance = df_relevance.sort_values(by='citedByCount', ascending=False).drop_duplicates(subset='title')
df_top = df_relevance.head(500)
df_top.to_excel('(AS+DB)_high_relevance.xlsx', index=False)



In [11]:
ref_id_list = []
df_top = pd.read_excel('(AS+DB)_high_relevance.xlsx', dtype={'pmid': str})
data = df_top['pmid'].to_list()
for one_data in data[:30]:
    if not one_data:
        continue
    pmid = one_data
    ref_list = get_references(pmid)
    ref_id_list += [ref.get('id',ref.get('title')) for ref in ref_list]
ref_id_list

['1128665',
 '27325748',
 '174715',
 '28344317',
 '28205560',
 '7275966',
 '1168101',
 '26318451',
 '14643656',
 '26121403',
 '22344696',
 '28106076',
 '27373337',
 '25219674',
 '16387656',
 '27613580',
 '10902565',
 '23604283',
 '28192787',
 '25569111',
 '25063673',
 '28176824',
 '17434869',
 '26343757',
 '25412661',
 '17187822',
 '21079590',
 '12114023',
 '26046440',
 '27302133',
 '23871666',
 '26578598',
 '27911188',
 '27376769',
 '25593312',
 '6318439',
 '24586927',
 '28320778',
 '25611135',
 '25242552',
 '13811056',
 '27558897',
 '14299636',
 '28052920',
 '26218273',
 '25799993',
 '856255',
 '28388414',
 '22417203',
 '26863410',
 '19059995',
 '25176256',
 '26075521',
 '26876937',
 '28334903',
 '4372599',
 '28495677',
 '26061753',
 '10580478',
 '18552770',
 '23352575',
 '25534324',
 '25780134',
 '22065625',
 '26458103',
 '24581449',
 '24407421',
 '26816380',
 '28230119',
 '22760636',
 '6201720',
 '27306184',
 '23812587',
 '26593424',
 '24981863',
 '26365242',
 '26321680',
 '2311848

In [12]:
from collections import Counter
counter = Counter(ref_id_list)
# 找出重复的元素及其数量
duplicates = {item: count for item, count in counter.items() if count > 1}

# 按数量降序排序
sorted_duplicates = sorted(duplicates.items(), key=lambda x: x[1], reverse=True)

print("重复的元素及其出现次数（降序）：")
for item, count in sorted_duplicates:
    print(f"{item}: {count}")


重复的元素及其出现次数（降序）：
23446348: 15
22319583: 14
23249747: 14
23446346: 14
7536344: 13
25242744: 12
24039610: 12
24811520: 11
25070500: 10
25242144: 10
25449546: 9
1991322: 9
25558066: 8
25664725: 8
1069269: 8
7684656: 8
21964070: 8
24035497: 8
28281539: 8
19213877: 8
25714049: 8
25921068: 8
25544350: 7
25543144: 7
22140119: 7
24609083: 7
28344082: 7
28344080: 7
25234927: 7
26593424: 6
7678559: 6
26861625: 6
28903484: 6
26046440: 5
20094052: 5
25281217: 5
25404635: 5
22955620: 5
28798046: 5
27050392: 5
25768908: 5
25624062: 5
4372599: 4
24407421: 4
24316715: 4
22575960: 4
23177736: 4
25192136: 4
22608085: 4
24284625: 4
25719671: 4
21593866: 4
22658674: 4
1339341: 4
8718689: 4
21151960: 4
23463100: 4
19167326: 4
20601959: 4
22056041: 4
24514441: 4
22388286: 4
26450910: 4
26138677: 4
460409: 4
8692851: 4
29980667: 4
26657634: 4
8566785: 4
28080204: 4
9740124: 4
27725737: 4
29343848: 4
27040497: 4
26873924: 4
26669964: 4
24014594: 4
26553571: 4
26121403: 3
22344696: 3
28106076: 3
27373337: 3
25

In [ ]:
#获得某一篇文献的所有引用的信息
def search_bypmid(pmid):
    url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/article/MED/{pmid}"
    params = {
        "format": "json",
        'resultType': 'core',
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()  # 确保HTTP 200
        article = response.json().get('result', {})
        
        dict_single = {
            'title': article.get('title', ''),
            'keywordList': article.get('keywordList', ''),
            'web': 'https://doi.org/' + article.get('doi', '') if article.get('doi') else '',
            'pubYear': article.get('pubYear', ''),
            'citedByCount': article.get('citedByCount', ''),
            'pmid': article.get('pmid', ''),
            'pmcid': article.get('pmcid', ''),
            'source': article.get('source', ''),
            'doi': article.get('doi', ''),
            'abstractText': article.get('abstractText', ''),
        }
        return dict_single, None

    except requests.exceptions.RequestException as e:
        return None, f"Request error for PMID {pmid}: {str(e)}"
    except json.JSONDecodeError:
        return None, f"JSON decode error for PMID {pmid}"
    except Exception as e:
        return None, f"Unexpected error for PMID {pmid}: {str(e)}"


ref_id_tuple = tuple(ref_id_list)
data_references = []
error_log = []

for pmid in ref_id_tuple:
    result, error = search_bypmid(pmid)
    if result:
        data_references.append(result)
    else:
        error_log.append(error)
        
df_ref = pd.DataFrame(data_references).drop_duplicates(subset='pmid')
df_ref.to_excel('all_references.xlsx', index=False)

# 保存错误日志
if error_log:
    with open('error_log.txt', 'w', encoding='utf-8') as f:
        for line in error_log:
            f.write(line + '\n')